# Calculate Query Metrics - 5 Row Sample

This notebook runs hard-coded submissions for GPT-4o, Claude, and Gemini on the five sampled rows, compares them to `golden_sample_0503/`, and overwrites `metrics.jsonl`, `facts.jsonl`, and `run_metrics.jsonl` with this sample run.

In [ ]:
import json
import queue
from pathlib import Path

import pandas as pd

from eval_runner import DEFAULT_CONNECTION_STRINGS
from golden_set import GoldenSet
from metric_calculator import MetricCalculator
from metric_manager import MetricManager
from metric_writer import MetricWriter
from query_runner import ParallelQueryRunner


In [ ]:
SAMPLE_RECORDS = [
    {
        "request_id": 25,
        "db_id": "Insurance",
        "question": "How many beneficiaries appear in inpatient claims but have no outpatient claims?",
        "golden_sql": "select count(distinct [DESYNPUF_ID]) as beneficiary from ( select [DESYNPUF_ID] from [dbo].[Inpatient_Claim] except select [DESYNPUF_ID] from [dbo].[Outpatient_Claim] ) as inpatient_only;",
        "models": {
            "gpt4o": "SELECT COUNT(DISTINCT b.DESYNPUF_ID) AS BeneficiaryCount\nFROM beneficiary b\nJOIN inpatient_claim ic ON b.DESYNPUF_ID = ic.DESYNPUF_ID\nLEFT JOIN outpatient_claim oc ON b.DESYNPUF_ID = oc.DESYNPUF_ID\nWHERE oc.DESYNPUF_ID IS NULL;",
            "claude": "SELECT COUNT(DISTINCT ic.DESYNPUF_ID) AS beneficiary_count\nFROM inpatient_claim ic\nWHERE ic.DESYNPUF_ID NOT IN (\n    SELECT DISTINCT DESYNPUF_ID\n    FROM outpatient_claim\n)",
            "gemini": "SELECT COUNT(DISTINCT T1.DESYNPUF_ID) FROM inpatient_claim AS T1 LEFT JOIN outpatient_claim AS T2 ON T1.DESYNPUF_ID = T2.DESYNPUF_ID WHERE T2.DESYNPUF_ID IS NULL"
        }
    },
    {
        "request_id": 29,
        "db_id": "Insurance",
        "question": "In 2010, how many beneficiaries meet at least one of these conditions:(1) Has diabetes and COPD, or (2) Has depression but no heart failure?",
        "golden_sql": "select count(distinct [DESYNPUF_ID]) as beneficiary from [dbo].[Beneficiary] where [YEAR]=2010 and ( ([SP_DIABETES]=1 and [SP_COPD]=1) or ([SP_DEPRESSN]=1 and [SP_CHF]=0) );",
        "models": {
            "gpt4o": "SELECT COUNT(DISTINCT b.DESYNPUF_ID) \nFROM beneficiary b \nWHERE b.YEAR = 2010 \nAND ((b.SP_DIABETES = 1 AND b.SP_COPD = 1) \nOR (b.SP_DEPRESSN = 1 AND b.SP_CHF = 2))",
            "claude": "\nSELECT COUNT(DISTINCT DESYNPUF_ID) AS beneficiary_count\nFROM beneficiary\nWHERE YEAR = 2010\n  AND (\n    (SP_DIABETES = 1 AND SP_COPD = 1)\n    OR (SP_DEPRESSN = 1 AND SP_CHF = 2)\n  )\n",
            "gemini": "SELECT COUNT(DISTINCT DESYNPUF_ID) FROM beneficiary WHERE YEAR = 2010 AND ((SP_DIABETES = 1 AND SP_COPD = 1) OR (SP_DEPRESSN = 1 AND SP_CHF = 2))"
        }
    },
    {
        "request_id": 93,
        "db_id": "GreaterManchesterCrime",
        "question": "What time do most of the crimes happen?",
        "golden_sql": "SELECT TOP (1) CrimeTS FROM GreaterManchesterCrime GROUP BY CrimeTS ORDER BY count(*) DESC;",
        "models": {
            "gpt4o": "SELECT TOP 1 CrimeTS, COUNT(*) AS CrimeCount\nFROM GreaterManchesterCrime\nGROUP BY CrimeTS\nORDER BY CrimeCount DESC;",
            "claude": "SELECT TOP 1 CrimeTS, COUNT(*) AS CrimeCount\nFROM GreaterManchesterCrime\nWHERE CrimeTS IS NOT NULL\nGROUP BY CrimeTS\nORDER BY CrimeCount DESC",
            "gemini": "SELECT TOP 1 Type, COUNT(*) FROM GreaterManchesterCrime GROUP BY Type ORDER BY COUNT(*) DESC"
        }
    },
    {
        "request_id": 166,
        "db_id": "TheHistoryofBaseball",
        "question": "Top 10 teams with the most hall of fame players",
        "golden_sql": "SELECT TOP (10) T2.team_id FROM hall_of_fame as T1 JOIN salary as T2 ON T1.player_id = T2.player_id AND T1.yearid = T2.year WHERE T1.inducted = 'Y' GROUP BY T2.team_id ORDER BY count(*) DESC;",
        "models": {
            "gpt4o": "SELECT TOP 10 s.team_id, COUNT(h.player_id) AS hall_of_fame_count\nFROM hall_of_fame h\nJOIN salary s ON h.player_id = s.player_id\nGROUP BY s.team_id\nORDER BY hall_of_fame_count DESC;",
            "claude": "SELECT TOP 10\n    s.team_id,\n    COUNT(DISTINCT h.player_id) AS hof_player_count\nFROM hall_of_fame h\nINNER JOIN salary s ON h.player_id = s.player_id\nWHERE h.inducted = 'Y'\nGROUP BY s.team_id\nORDER BY hof_player_count DESC",
            "gemini": "SELECT TOP 10 T2.team_id ,  count(*) FROM hall_of_fame AS T1 JOIN salaries AS T2 ON T1.player_id = T2.player_id GROUP BY T2.team_id ORDER BY count(*) DESC"
        }
    },
    {
        "request_id": 237,
        "db_id": "WhatCDHipHop",
        "question": "Find me the most popular topics since 2010.",
        "golden_sql": "SELECT TOP (10) T1.tag FROM torrents as T2 JOIN tags as T1 ON T1.id = T2.id WHERE T2.groupYear >= 2010 GROUP BY T1.tag ORDER BY T2.totalSnatched DESC;",
        "models": {
            "gpt4o": "SELECT tag, COUNT(*) AS popularity\nFROM tags\nJOIN torrents ON tags.id = torrents.groupId\nWHERE torrents.groupYear >= 2010\nGROUP BY tag\nORDER BY popularity DESC;",
            "claude": "SELECT TOP 10 t.tag, COUNT(*) as frequency\nFROM tags t\nINNER JOIN torrents tr ON t.index = tr.id\nWHERE tr.groupYear >= 2010\nGROUP BY t.tag\nORDER BY frequency DESC",
            "gemini": "SELECT TOP 10 groupName, totalSnatched FROM torrents ORDER BY totalSnatched DESC"
        }
    }
]

[(r["request_id"], r["db_id"], r["question"]) for r in SAMPLE_RECORDS]


In [ ]:
GOLDEN_OUT_DIR = "golden_sample_0503"
FACTS_JSONL = "facts.jsonl"
METRICS_JSONL = "metrics.jsonl"
RUN_METRICS_JSONL = "run_metrics.jsonl"

for path in [FACTS_JSONL, METRICS_JSONL, RUN_METRICS_JSONL]:
    Path(path).write_text("", encoding="utf-8")

golden = GoldenSet(out_dir=GOLDEN_OUT_DIR)
metric_calculator = MetricCalculator(golden)


In [ ]:
def run_model_sample(records):
    events_q = queue.Queue()
    writer = MetricWriter(
        event_queue=events_q,
        connection_string=None,
        facts_jsonl_path=FACTS_JSONL,
        run_metrics_jsonl_path=RUN_METRICS_JSONL,
        metrics_jsonl_path=METRICS_JSONL,
    )
    writer.start()

    mm = MetricManager(writer_queue=events_q, run_label="sample-0503")
    mm.register_per_query_metric(metric_calculator.accuracy_vs_golden)
    mm.register_per_query_metric(metric_calculator.runtime_delta_vs_golden)
    mm.enqueue_run_header(notes="5-row hard-coded sample from master_results_with_submit_edited_0503.xlsx")

    for db_id in sorted({r["db_id"] for r in records}):
        db_records = [r for r in records if r["db_id"] == db_id]
        runner = ParallelQueryRunner(
            connection_string=DEFAULT_CONNECTION_STRINGS[db_id],
            num_workers=2,
            result_callback=mm.add_result,
        )
        runner.start()
        for r in db_records:
            for model, sql in r["models"].items():
                runner.submit(
                    sql,
                    metadata={
                        "run": "sample_0503",
                        "model": model,
                        "request_id": r["request_id"],
                        "db_id": r["db_id"],
                        "question": r["question"],
                    },
                )
        runner.stop(drain=True)

    mm.flush_writes()
    writer.stop(drain=True)
    return writer.get_stats()


writer_stats = run_model_sample(SAMPLE_RECORDS)
writer_stats


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

metrics = load_jsonl(METRICS_JSONL)
facts = load_jsonl(FACTS_JSONL)

metrics_df = pd.DataFrame(metrics)
summary = {}
if not metrics_df.empty:
    enriched = []
    for rec in metrics:
        md = rec.get("metadata") or {}
        enriched.append({
            "request_id": md.get("request_id"),
            "db_id": md.get("db_id"),
            "model": md.get("model"),
            "metric_name": rec.get("metric_name"),
            "metric_value": rec.get("metric_value"),
        })
    long_df = pd.DataFrame(enriched)
    wide = long_df.pivot_table(
        index=["request_id", "db_id", "model"],
        columns="metric_name",
        values="metric_value",
        aggfunc="first",
    ).reset_index()

    for col in ["exact_match", "row_count_match", "runtime_delta_ms", "runtime_delta_pct"]:
        if col in wide.columns:
            wide[col] = pd.to_numeric(wide[col], errors="coerce")

    def finite_or_none(value):
        value = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
        return None if pd.isna(value) else float(value)

    by_model = []
    for model, group in wide.groupby("model"):
        by_model.append({
            "model": model,
            "query_count": int(len(group)),
            "exact_match_rate": finite_or_none(group["exact_match"].mean()) if "exact_match" in group else None,
            "row_count_match_rate": finite_or_none(group["row_count_match"].mean()) if "row_count_match" in group else None,
            "avg_runtime_delta_ms": finite_or_none(group["runtime_delta_ms"].mean()) if "runtime_delta_ms" in group else None,
            "avg_runtime_delta_pct": finite_or_none(group["runtime_delta_pct"].mean()) if "runtime_delta_pct" in group else None,
        })

    summary = {
        "run_label": "sample-0503",
        "sample_request_ids": [r["request_id"] for r in SAMPLE_RECORDS],
        "fact_count": len(facts),
        "metric_event_count": len(metrics),
        "summary_by_model": by_model,
    }

with open(RUN_METRICS_JSONL, "w", encoding="utf-8") as f:
    f.write(json.dumps(summary, ensure_ascii=False, default=str, allow_nan=False) + "\n")

summary
